In [0]:
%pip install Faker
%pip install --upgrade typing-extensions
%pip install pybaseball


In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql import SparkSession
import datetime
import re
import os
import time
import pyspark
from pyspark.sql.types import StructType, StructField, BooleanType, DoubleType, LongType
from pyspark.sql.types import StringType
from pyspark.sql.types import DateType
from pyspark.sql.types import IntegerType
from pyspark.sql import Row
from datetime import date
from faker import Faker
import random
import numpy as np
from pybaseball import statcast, cache
from datetime import timedelta



In [0]:
def get_dates_between(start_date, end_date):
    delta = end_date - start_date         # Calculate the difference in days
    dates_list = []
    for i in range(delta.days + 1):       # Iterate over the number of days, inclusive
        day = start_date + timedelta(days=i)
        dates_list.append(day)

    
    return dates_list

In [0]:
def convert_dt_list_to_strings(dt_list):

    rt_list = []
    for elem in dt_list:
        elem = elem.strftime("%Y-%m-%d")
        rt_list.append(elem)
    return rt_list

In [0]:
def load_baseball_savant_data(START_DATE: str, END_DATE: str):
    savant_data = statcast(START_DATE, END_DATE)


    return savant_data

In [0]:
cache.enable()

In [0]:
dt_list = get_dates_between(date(2022, 5, 1), date(2022, 5, 20))


dt_list_str = convert_dt_list_to_strings(dt_list)



print(dt_list_str)


In [0]:

spark = SparkSession.builder \
        .appName('Autoloader Example w/ CSVs') \
                .master("local[*]") \
    .config("spark.driver.memory", "9g") \
    .getOrCreate()






#spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")
spark.conf.set("spark.sql.session.timeZone", "UTC")
#spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
#spark._jsc.hadoopConfiguration().set(f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",f'{STORAGE_ACCOUNT_KEY}')

In [0]:
i = 0

for day in dt_list_str:
    df = load_baseball_savant_data(day, day)


    spark_df = spark.createDataFrame(df)


    spark_df \
        .write\
        .format("csv") \
        .mode("overwrite") \
        .option("header", "true") \
        .option("delimiter", ",") \
        .save(f"/Volumes/mlb_demo/default/file_examples/savant_csvs/pbp_data_{i}.csv")

    i += 1



#     df = load_baseball_savant_data(dt_list_str[0]

In [0]:
data_df = spark \
            .read\
            .format('csv') \
            .option("header", "true") \
            .option("delimiter", ",") \
            .load(f"/Volumes/mlb_demo/default/file_examples/savant_csvs/pbp_data_*.csv")

data_df.write.mode("overwrite").saveAsTable("mlb_demo.default.sv_data")

In [0]:
%sql


SELECT * FROM mlb_demo.default.sv_data

## Set up an Auto Stream

In [0]:
spark \
        .readStream.format('cloudFiles') \
        .option("cloudFiles.format", "csv") \
        .option("cloudFiles.schemaLocation", "/Volumes/mlb_demo/default/file_examples/schemas/savant") \
        .load("/Volumes/mlb_demo/default/file_examples/savant_csvs/pbp_data_*.csv")\
        .writeStream\
        .option("checkpointLocation", "/Volumes/mlb_demo/default/file_examples/checkpoint/")\
        .toTable("mlb_demo.default.sv_data_stream")

In [0]:
%sql

SELECT COUNT(*) FROM mlb_demo.default.sv_data_stream

## Insert More Data to the stream

In [0]:
i = 20

dt_list = get_dates_between(date(2022, 5, 21), date(2022, 5, 30))


dt_list_str = convert_dt_list_to_strings(dt_list)



for day in dt_list_str:
    df = load_baseball_savant_data(day, day)


    spark_df = spark.createDataFrame(df)


    spark_df \
        .write\
        .format("csv") \
        .mode("overwrite") \
        .option("header", "true") \
        .option("delimiter", ",") \
        .save(f"/Volumes/mlb_demo/default/file_examples/savant_csvs/pbp_data_{i}.csv")

    i += 1



#     df = load_baseball_savant_data(dt_list_str[0]